# XP Exercise — Cats vs Dogs Binary Image Classification

**Course:** Developers Institute  **Week 6 - Day 4**  
**Author:** Alex Goldbaum

End-to-end binary image classification: a small CNN trained on the Kaggle
Dogs vs Cats dataset, with **data augmentation**, **baseline ablation**,
**class-imbalance handling**, **inference on the unlabeled test split** and
persistent artifacts. Twelve sequential sections following the task brief.

**Before running**: place the extracted dataset at `data/cats_dogs/`:
```
data/cats_dogs/
    train/train/cat.0.jpg ... dog.12499.jpg
    test/test/1.jpg ... 12500.jpg
```
On Colab: upload the ZIP, then `!unzip cats_dogs.zip -d data/` and rename if needed.


## 0. Setup


In [ ]:
%pip install -qU tensorflow pyyaml


In [ ]:
# Prefilled. Just copy and execute.
import os, math, re, random
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

np.random.seed(42); tf.random.set_seed(42)

# Paths - change if needed
DATA_ROOT = Path('data/cats_dogs')
train_dir = (DATA_ROOT / 'train' / 'train') if (DATA_ROOT / 'train' / 'train').exists() else (DATA_ROOT / 'train')
test_dir  = (DATA_ROOT / 'test'  / 'test')  if (DATA_ROOT / 'test'  / 'test').exists()  else (DATA_ROOT / 'test')

IMG_HEIGHT, IMG_WIDTH = 180, 180
batch_size = 32
seed = 1337

# Build DataFrames from folders
def build_df_from_folder(folder: Path, labeled: bool=True):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        raise FileNotFoundError(f'No images found under {folder}')
    rows = []
    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {'cat','cats'}:
                label = 'cat'
            elif parent in {'dog','dogs'}:
                label = 'dog'
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name): label = 'cat'
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = 'dog'
                else:
                    continue
            rows.append({'filepath': f, 'label': label})
        else:
            rows.append({'filepath': f})
    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full  = build_df_from_folder(test_dir,  labeled=False)

# Train validation split
from sklearn.model_selection import train_test_split
df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2, stratify=df_train_full['label'], random_state=seed
)

# Generators
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True,
)
val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=batch_size,
    shuffle=True, seed=seed, validate_filenames=False,
)
val_flow = val_gen.flow_from_dataframe(
    df_val, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=batch_size,
    shuffle=False, validate_filenames=False,
)
test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col='filepath', y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None, batch_size=batch_size,
    shuffle=False, validate_filenames=False,
)

print({'train': train_flow.samples, 'val': val_flow.samples, 'test': test_flow.samples,
       'class_indices': train_flow.class_indices})


## 2. Inspect the Data


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

# Class counts from the training set
class_counts = df_tr['label'].value_counts()
print('Training class counts:')
print(class_counts)
print(f'\nClass indices: {train_flow.class_indices}')

# Visualize the balance
plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=['steelblue', 'tomato'])
plt.title('Training images per class', fontweight='bold')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

ratio = class_counts.max() / class_counts.min()
print(f'Imbalance ratio: {ratio:.2f}  ->',
      'BALANCED' if ratio < 1.1 else 'IMBALANCED — consider class_weight')


**Class balance analysis.** The Kaggle Dogs vs Cats training set is
**perfectly balanced** (12,500 cats / 12,500 dogs). With class balance this
clean we do not need `class_weight` for the main run — accuracy and ROC-AUC
are both honest metrics. We still implement the class-weighted version later
as Section 9 requires.

**Sources of visual variability.** Pose (sitting, standing, crouched), scale
(close-ups vs full body), lighting (indoor flash, outdoor sun), background
(carpets, gardens, with other animals or humans), occlusion (partially
hidden), colour distribution (white cats, black dogs, etc.), and viewpoint
(frontal vs profile). Augmentation has to be wide enough to expose the model
to these variations without invalidating the label.


In [ ]:
# Image grid: 8 cats + 8 dogs
samples = (
    df_tr[df_tr['label'] == 'cat'].sample(8, random_state=seed).assign(label='cat'),
    df_tr[df_tr['label'] == 'dog'].sample(8, random_state=seed).assign(label='dog'),
)
sample_df = pd.concat(samples).reset_index(drop=True)

fig, axes = plt.subplots(4, 4, figsize=(11, 11))
for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    img = tf.keras.utils.load_img(row['filepath'], target_size=(IMG_HEIGHT, IMG_WIDTH))
    ax.imshow(img)
    color = 'steelblue' if row['label'] == 'cat' else 'tomato'
    ax.set_title(row['label'], color=color, fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample training images', fontweight='bold')
plt.tight_layout()
plt.show()


**Visual cues to discriminate.** Cats: pointed ears, vertical pupils, smaller
body proportions, often whiskers visible. Dogs: floppy or alert ears,
longer snout, bigger overall body, distinct breed silhouettes. Texture and
fur patterns also matter — but colour alone is not enough (both species cover
the full colour range).


## 3. CNN Architecture

**Planned architecture.** A small VGG-style stack: **3 convolutional blocks**
with progressively more filters (32 → 64 → 128). Each block has
`Conv2D(3×3, padding='same', ReLU) → BatchNormalization → MaxPooling2D(2×2)`.
`Same` padding keeps the spatial dimensions during convolutions; `MaxPooling`
halves them between blocks so the receptive field grows. After the
convolutional stack we `Flatten`, pass through a `Dense(128, ReLU)` head with
`Dropout(0.5)` to regularize the wide dense layer, and end with a single
**sigmoid unit** — the correct output for a binary Bernoulli target. The
matching loss is **binary cross-entropy**.


In [ ]:
from tensorflow.keras import layers, models

def build_cnn(img_h=IMG_HEIGHT, img_w=IMG_WIDTH, dropout=0.5):
    m = models.Sequential([
        layers.Input(shape=(img_h, img_w, 3)),

        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(1, activation='sigmoid'),  # binary output
    ])
    return m


model = build_cnn()
model.summary()


## 4. Optimization Setup

**Justification.**
- **Optimizer:** Adam(`lr=1e-3`) — adaptive learning rate, fast and robust
  defaults for image tasks.
- **Initial LR `1e-3`:** Adam's default; high enough to learn quickly with
  BatchNorm, low enough not to diverge. We let `ReduceLROnPlateau` shrink it
  when validation loss stalls.
- **Batch size 32:** fits comfortably on Colab GPU memory for 180×180 RGB
  images. Larger batches give smoother gradients but can hurt generalization;
  smaller batches are noisier.
- **EarlyStopping(patience=4, restore_best_weights=True)** on `val_loss`
  prevents wasted epochs once the model stops improving.
- **ReduceLROnPlateau(factor=0.5, patience=2)** halves the LR after two
  stalled epochs — a cheap rescue when the optimizer gets stuck on a plateau.

We monitor **both loss and accuracy**: accuracy is intuitive but coarser;
loss is smoother and reflects probability quality.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1),
]


## 5. Train the Model

Train with the augmented `train_flow` and validate on the clean `val_flow`.
We then plot the loss/accuracy curves and look for the classic overfitting
signal: training loss continues to drop while validation loss flattens or rises.


In [ ]:
EPOCHS = 15

history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Binary CE'); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Best epoch (val_loss): {int(np.argmin(history.history['val_loss'])) + 1}")


**Detecting overfitting.** When `train_loss` keeps falling while `val_loss`
plateaus or rises, the model is memorising the training set. Mitigations we
already wired in: heavy augmentation in `train_flow`, dropout(0.5) in the
dense head, early stopping with weight restore, and learning-rate decay on
plateau. If overfitting still dominates, the next levers would be stronger
augmentation (`zoom_range`, `shear_range`), a higher dropout, or a smaller
convolutional stack.


## 6. Evaluate on Validation Data


In [ ]:
val_loss, val_acc = model.evaluate(val_flow, verbose=0)
print(f'Validation loss: {val_loss:.4f}')
print(f'Validation acc : {val_acc:.4f}')


In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report, precision_score, recall_score,
)

val_flow.reset()
y_proba = model.predict(val_flow, verbose=0).ravel()
y_pred = (y_proba > 0.5).astype(int)
y_true = val_flow.classes

labels_inv = {v: k for k, v in train_flow.class_indices.items()}
target_names = [labels_inv[0], labels_inv[1]]

cm = confusion_matrix(y_true, y_pred)
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion matrix — validation', fontweight='bold')
plt.tight_layout()
plt.show()


**Reading the errors.** Compare the false-positive and false-negative counts
in the confusion matrix. If one error type dominates (e.g., many cats
predicted as dogs), the model has a bias toward that class — possibly
because of colour or texture cues that generalise poorly. Two easy fixes:
**(1)** shift the decision threshold away from 0.5 to balance precision /
recall on the metric that matters; **(2)** add more targeted augmentation
for the under-detected class.


## 7. Inference on the Unlabeled Test Set

Predict probabilities on `test_flow`, choose a threshold (`0.5` is fine on a
balanced dataset — would lower it if false negatives on dogs were more
costly), and export a CSV with `filepath`, `prob_dog`, `pred_label`.


In [ ]:
test_flow.reset()
test_proba = model.predict(test_flow, verbose=0).ravel()

# Convention: keras assigns 0='cat', 1='dog' alphabetically, so the sigmoid output
# directly represents P(dog).
dog_idx = train_flow.class_indices['dog']
if dog_idx == 0:
    test_proba = 1.0 - test_proba

THRESHOLD = 0.5
pred_labels = np.where(test_proba >= THRESHOLD, 'dog', 'cat')

test_df = pd.DataFrame({
    'filepath': df_test_full['filepath'].values,
    'prob_dog': test_proba,
    'pred_label': pred_labels,
})
out_csv = 'test_predictions.csv'
test_df.to_csv(out_csv, index=False)
print(f'Saved {len(test_df)} predictions to {out_csv}')
test_df.head()


In [ ]:
# Manual sanity check: 6 most confident dogs, 6 most confident cats, 4 most uncertain
most_dog = test_df.nlargest(6, 'prob_dog')
most_cat = test_df.nsmallest(6, 'prob_dog')
most_unsure = test_df.iloc[(test_df['prob_dog'] - 0.5).abs().argsort()].head(4)

fig, axes = plt.subplots(4, 4, figsize=(13, 13))
groups = [('Most confident DOG', most_dog),
          ('Most confident CAT', most_cat),
          ('Most uncertain',     most_unsure)]
all_rows = pd.concat([most_dog, most_cat, most_unsure]).reset_index(drop=True)
for ax, (_, row) in zip(axes.flat, all_rows.iterrows()):
    img = tf.keras.utils.load_img(row['filepath'], target_size=(IMG_HEIGHT, IMG_WIDTH))
    ax.imshow(img)
    ax.set_title(f"{row['pred_label']} (P_dog={row['prob_dog']:.2f})", fontsize=10)
    ax.axis('off')
plt.suptitle('Manual review — confident + uncertain test predictions',
             fontweight='bold')
plt.tight_layout()
plt.show()


**Manual verification strategy.** We always sample the *most confident* and
the *most uncertain* test predictions for a human pass before trusting the
model. Confident-but-wrong cases reveal systematic biases (e.g., the model
labels every white-furred animal as cat). Uncertain cases pinpoint the kinds
of input where the model needs more data or stronger features.


## 8. Baseline vs Augmentation Ablation

Train the same architecture but with **no augmentation** to see the lift
from augmentation alone. Same epochs, same callbacks, same val split — only
the train generator changes.


In [ ]:
# Baseline train generator: only rescale, no augmentation
baseline_train_gen = ImageDataGenerator(rescale=1./255)
baseline_flow = baseline_train_gen.flow_from_dataframe(
    df_tr, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=batch_size,
    shuffle=True, seed=seed, validate_filenames=False,
)

tf.random.set_seed(42)
baseline_model = build_cnn()
baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
history_baseline = baseline_model.fit(
    baseline_flow,
    validation_data=val_flow,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=0,
)

base_val_loss, base_val_acc = baseline_model.evaluate(val_flow, verbose=0)
print(f'Baseline (no aug) — val loss: {base_val_loss:.4f}  val acc: {base_val_acc:.4f}')
print(f'Augmented         — val loss: {val_loss:.4f}  val acc: {val_acc:.4f}')
print(f'Lift from augmentation: +{(val_acc - base_val_acc)*100:.2f} pp val acc')


In [ ]:
# Compare learning curves side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history.history['val_accuracy'], label='Augmented', marker='o')
axes[0].plot(history_baseline.history['val_accuracy'], label='Baseline', marker='s')
axes[0].set_title('Validation accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

axes[1].plot(history.history['val_loss'], label='Augmented', marker='o')
axes[1].plot(history_baseline.history['val_loss'], label='Baseline', marker='s')
axes[1].set_title('Validation loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Binary CE'); axes[1].legend()
plt.tight_layout()
plt.show()


**Generalization gap analysis.** The baseline typically overfits faster: the
train/val gap widens early because the model can memorise the (unaltered)
training images. The augmented model sees a slightly different image every
epoch, so memorisation is harder — train accuracy climbs more slowly but val
accuracy holds up further into training. Net effect: better generalization
with no extra parameters.


## 9. Class Imbalance Handling (defensive)

Our dataset is balanced, but here is how we would handle imbalance.
`class_weight` rescales the loss contribution of each class so the network
does not collapse onto the majority class. We compute weights with
`compute_class_weight('balanced', ...)` and pass them to `model.fit`.


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(df_tr['label'])
class_arr = df_tr['label'].map(train_flow.class_indices).values
weights = compute_class_weight('balanced', classes=np.unique(class_arr), y=class_arr)
class_weight = dict(zip(np.unique(class_arr).tolist(), weights.tolist()))
print('Class weights:', class_weight)


**Expected effect on the minority class.** With `class_weight`, the network
pays more for misclassifying the rarer class — its **recall** rises (we
catch more of those examples) at the cost of **precision** (more false
positives). The trade-off can be tuned by adjusting the weights or the
decision threshold. On a perfectly balanced dataset the weights are both
`1.0` and the result is identical to unweighted training.


## 10. Save Artifacts for Reuse

Saving **both** the weights and a metadata file (image size, class indices,
training config) is necessary because the weights alone are not enough to
rebuild a working inference pipeline: at the very least we need the input
shape, the class index mapping, and the preprocessing settings.


In [ ]:
import yaml

OUT_DIR = Path('artifacts')
OUT_DIR.mkdir(exist_ok=True)

model.save(OUT_DIR / 'cats_dogs_cnn.h5')
model.save(OUT_DIR / 'cats_dogs_savedmodel')

metadata = {
    'task': 'cats_vs_dogs_binary',
    'img_height': IMG_HEIGHT,
    'img_width': IMG_WIDTH,
    'batch_size': batch_size,
    'class_indices': {k: int(v) for k, v in train_flow.class_indices.items()},
    'augmentation': {
        'rotation_range': 45,
        'width_shift_range': 0.15,
        'height_shift_range': 0.15,
        'zoom_range': 0.5,
        'horizontal_flip': True,
    },
    'optimizer': 'Adam',
    'learning_rate': 1e-3,
    'loss': 'binary_crossentropy',
    'epochs_trained': len(history.history['loss']),
    'final_val_accuracy': float(val_acc),
    'final_val_loss': float(val_loss),
}
with open(OUT_DIR / 'config.yaml', 'w') as f:
    yaml.safe_dump(metadata, f, sort_keys=False)

print('Saved artifacts:')
for p in OUT_DIR.rglob('*'):
    if p.is_file():
        print(' -', p, f'({p.stat().st_size / 1024:.1f} KB)')


## 11. Extension — Transfer Learning with MobileNetV2

**Proposal.** Replace the small from-scratch CNN with a **MobileNetV2 backbone**
(frozen, ImageNet-pretrained) plus a small classifier head: `GlobalAveragePooling2D`
→ `Dropout(0.3)` → `Dense(1, sigmoid)`.

**Expected benefit.** ImageNet-pretrained features are excellent priors for
natural-image tasks: edges, textures, shapes generalize across object
classes. With the backbone frozen we train only ~2,000 parameters in the
head, so it fits in seconds even on a CPU and typically reaches **>96% val
accuracy** in 3–5 epochs — versus the ~85-92% the from-scratch model takes
15 epochs to reach. A second 'fine-tune' pass that unfreezes the top
MobileNet block can squeeze out another 1–2 percentage points.

Quick code sketch (run as an extension experiment if time permits):


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def build_transfer_model(img_h=IMG_HEIGHT, img_w=IMG_WIDTH):
    base = MobileNetV2(input_shape=(img_h, img_w, 3), include_top=False, weights='imagenet')
    base.trainable = False
    inputs = layers.Input(shape=(img_h, img_w, 3))
    x = layers.Rescaling(255.0)(inputs)            # undo our /255
    x = preprocess_input(x)                        # apply MobileNet's own preproc
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inputs, outputs)


tl_model = build_transfer_model()
tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
print(f'Transfer model trainable params: {sum(p.numpy().size for p in tl_model.trainable_weights):,}')
# Train with: tl_model.fit(train_flow, validation_data=val_flow, epochs=5, callbacks=callbacks)


## 12. Deliverables Checklist

| Deliverable | Where in this notebook |
|---|---|
| Data report — class counts + sample grid | Section 2 |
| Model description + optimization rationale | Sections 3 + 4 |
| Training and validation curves with interpretation | Section 5 |
| Validation metrics — confusion matrix + precision/recall | Section 6 |
| Test predictions CSV with probabilities and labels | Section 7 (`test_predictions.csv`) |
| Saved model + run log | Section 10 (`artifacts/cats_dogs_cnn.h5`, `artifacts/config.yaml`) |

**Quick reminders honoured.**
- ✅ Augmentation applied **only** to the training generator.
- ✅ Validation generator uses `rescale` only; test generator uses `rescale` only.
- ✅ Validation split is **fixed** (`random_state=seed`) so experiments are comparable.
- ✅ Data paths and the input pipeline are validated *before* any modeling.
